In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os
import numpy as np
from tqdm import tqdm

In [32]:
model = models.resnet50(pretrained=True)

# Remove final classification layer
model = nn.Sequential(*list(model.children())[:-1])
model.eval()



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [33]:
df = pd.read_csv("/content/drive/MyDrive/WIRELESSCOM/wcomdataset.csv")

In [34]:
df = df.drop(columns=["Unnamed: 0"])

In [35]:
df

,index,Longitude,Latitude,Speed,Distance,Distance_x,Distance_y,PCI_64,PCI_65,PCI_302,RSRP
0,53735,12.513810,55.781135,15.88,0.721475,-0.003535,-0.009671,0,0,1,-124.83
1,56186,12.528917,55.795474,0.87,1.248863,0.010804,0.005436,0,0,1,-136.53
2,52546,12.527072,55.795562,13.57,1.232131,0.010892,0.003591,0,0,1,-133.63
3,54201,12.529027,55.792470,34.42,0.934332,0.007800,0.005546,0,0,1,-135.14
4,56261,12.528338,55.790267,54.89,0.692687,0.005597,0.004857,0,0,1,-123.17
...,...,...,...,...,...,...,...,...,...,...,...
795,19398,12.521089,55.785623,9.73,0.183340,0.000953,-0.002392,0,1,0,-68.78
796,18225,12.516981,55.785333,16.35,0.413170,0.000663,-0.006500,0,1,0,-64.80
797,19996,12.521737,55.785450,21.74,0.139360,0.000780,-0.001744,0,1,0,-68.96
798,173,12.520005,55.779501,17.57,0.614704,-0.005169,-0.003476,1,0,0,-64.23


In [38]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [36]:
def extract_features(df, image_folder):

    num_samples = len(df)
    features = np.zeros((num_samples, 2048))

    valid_rows = []

    for i, idx in enumerate(tqdm(df['index'])):
        img_path = os.path.join(image_folder, str(int(idx)) + ".png")

        try:
            img = Image.open(img_path).convert("RGB")
            img = transform(img).unsqueeze(0)   # no .to(device)

            with torch.no_grad():
                feat = model(img)
                feat = feat.squeeze().numpy()

            features[i] = feat
            valid_rows.append(i)

        except Exception as e:
            print(f"Skipping {idx}: {e}")

    # keep only valid rows
    features = features[valid_rows]
    df_filtered = df.iloc[valid_rows].reset_index(drop=True)

    return features, df_filtered

In [40]:
image_folder = "/content/drive/MyDrive/WIRELESSCOM/images/"   # CHANGE THIS

img_features, df_filtered = extract_features(df, image_folder)

print("Image features shape:", img_features.shape)


100%|██████████| 800/800 [03:00<00:00,  4.42it/s]

Image features shape: (800, 2048)


In [44]:
dfimg = pd.DataFrame(img_features)

In [45]:
dfimg

,0,1,2,3,4,5,6,7,8,9,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
0,0.281046,0.241069,0.138217,0.896623,0.075369,0.007972,0.108291,0.646822,0.437680,1.890079,...,0.175641,0.161646,0.126639,0.259446,0.385074,0.189437,0.349370,0.013486,0.027052,0.013935
1,0.678242,0.175652,1.890687,1.076171,0.074133,0.470325,0.491576,1.291971,0.457454,1.966635,...,0.194241,0.314547,0.035751,0.675712,0.352256,0.308011,0.514538,0.046220,0.135227,0.252701
2,0.175750,0.267848,0.863636,0.937123,0.311284,0.359959,0.465059,1.284342,0.179028,1.353485,...,0.276409,0.544446,0.033498,0.341831,0.210568,0.201642,0.496830,0.015991,0.122844,0.161192
3,0.326693,0.269951,0.518544,1.154887,0.010990,0.416268,0.106664,0.737561,0.378513,2.166049,...,0.501699,0.017719,0.248106,1.734224,0.293716,0.069073,0.266158,0.282083,0.336588,0.093251
4,0.574287,0.116039,0.294125,0.245566,0.110433,0.582795,0.042423,0.340572,0.755353,1.037750,...,0.266196,0.790590,0.158744,1.727435,0.501546,0.018821,0.405196,0.036183,0.660897,0.140315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,0.213394,0.156593,0.374242,0.677381,0.079260,0.139701,0.180441,0.972088,0.182031,1.802789,...,0.088299,0.340230,0.027657,0.560509,0.124379,0.021747,0.206987,0.000557,0.051594,0.037624
796,0.097684,0.050974,0.211388,0.893703,0.221635,0.105585,0.205586,1.361661,0.350437,0.700490,...,0.203582,0.224395,0.025778,0.721136,0.180133,0.123436,0.555273,0.149298,0.053516,0.049019
797,0.164940,0.434281,0.357654,0.578764,0.036939,0.007302,0.225197,0.575146,0.219311,1.175878,...,0.014433,0.052419,0.102437,0.622299,0.036121,0.086021,0.083251,0.002108,0.075019,0.077224
798,0.260377,0.134850,0.315243,0.748309,0.126093,0.104564,0.162161,1.199053,0.141245,1.089695,...,0.162576,0.624209,0.368652,0.572670,0.162509,0.071648,0.273291,0.290317,0.070053,0.053627


In [46]:
dfnew = pd.concat([df, dfimg], axis=1)

In [47]:
dfnew

,index,Longitude,Latitude,Speed,Distance,Distance_x,Distance_y,PCI_64,PCI_65,PCI_302,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
0,53735,12.513810,55.781135,15.88,0.721475,-0.003535,-0.009671,0,0,1,...,0.175641,0.161646,0.126639,0.259446,0.385074,0.189437,0.349370,0.013486,0.027052,0.013935
1,56186,12.528917,55.795474,0.87,1.248863,0.010804,0.005436,0,0,1,...,0.194241,0.314547,0.035751,0.675712,0.352256,0.308011,0.514538,0.046220,0.135227,0.252701
2,52546,12.527072,55.795562,13.57,1.232131,0.010892,0.003591,0,0,1,...,0.276409,0.544446,0.033498,0.341831,0.210568,0.201642,0.496830,0.015991,0.122844,0.161192
3,54201,12.529027,55.792470,34.42,0.934332,0.007800,0.005546,0,0,1,...,0.501699,0.017719,0.248106,1.734224,0.293716,0.069073,0.266158,0.282083,0.336588,0.093251
4,56261,12.528338,55.790267,54.89,0.692687,0.005597,0.004857,0,0,1,...,0.266196,0.790590,0.158744,1.727435,0.501546,0.018821,0.405196,0.036183,0.660897,0.140315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,19398,12.521089,55.785623,9.73,0.183340,0.000953,-0.002392,0,1,0,...,0.088299,0.340230,0.027657,0.560509,0.124379,0.021747,0.206987,0.000557,0.051594,0.037624
796,18225,12.516981,55.785333,16.35,0.413170,0.000663,-0.006500,0,1,0,...,0.203582,0.224395,0.025778,0.721136,0.180133,0.123436,0.555273,0.149298,0.053516,0.049019
797,19996,12.521737,55.785450,21.74,0.139360,0.000780,-0.001744,0,1,0,...,0.014433,0.052419,0.102437,0.622299,0.036121,0.086021,0.083251,0.002108,0.075019,0.077224
798,173,12.520005,55.779501,17.57,0.614704,-0.005169,-0.003476,1,0,0,...,0.162576,0.624209,0.368652,0.572670,0.162509,0.071648,0.273291,0.290317,0.070053,0.053627


In [48]:
dffinal = dfnew.drop(columns=['index'])

In [52]:
dffinal['RSRP']

,RSRP
0,-124.83
1,-136.53
2,-133.63
3,-135.14
4,-123.17
...,...
795,-68.78
796,-64.80
797,-68.96
798,-64.23


In [53]:
dffinal.to_csv('/content/drive/MyDrive/WIRELESSCOM/final_dataset.csv')